# Run Combined Image-to-3D Pipeline with Authentication on Modal

This notebook deploys the combined `image-to-3d` pipeline (Background removal + 3D generation) and queries its secure web endpoint with proxy authentication.

### Step 1: Deploy the Model

In [ ]:
# Deploy the pipeline model to Modal
!uv run modal deploy ../llm-hosting/image-to-3d.py

### Step 2: Load Credentials and Setup Endpoint

In [1]:
import os
import requests
import base64

def load_dotenv():
    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        v = v.strip().strip("'\"")
                        os.environ.setdefault(k.strip(), v)
            break

load_dotenv()
MODAL_KEY = os.environ.get("MODAL_KEY")
MODAL_SECRET = os.environ.get("MODAL_SECRET")

if not MODAL_KEY or not MODAL_SECRET:
    print("WARNING: Credentials not loaded!")
else:
    print("Credentials loaded successfully!")

username = "sshibinthomass"
ENDPOINT_URL = f"https://{username}--image-to-3d-imageto3d-generate-api.modal.run"
print(f"Target Endpoint URL: {ENDPOINT_URL}")

Credentials loaded successfully!
Target Endpoint URL: https://sshibinthomass--image-to-3d-imageto3d-generate-api.modal.run


### Step 3: Run Authenticated Inference

In [2]:
def test_inference(image_path: str, seed: int = 42):
    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")
        
    headers = {
        "Content-Type": "application/json",
        "Modal-Key": MODAL_KEY,
        "Modal-Secret": MODAL_SECRET,
    }
    payload = {
        "image_base64": image_base64,
        "seed": seed,
        "pipeline_type": "1024_cascade",
        "decimation_target": 1000000,
        "texture_size": 4096,
    }
    
    print("Sending request to Image-to-3D secure endpoint...")
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.content
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

from pathlib import Path
from datetime import datetime

image_path = "../llm-inference/img5.png"
output_bytes = test_inference(image_path)

if output_bytes:
    os.makedirs("outputs", exist_ok=True)
    
    input_stem = Path(image_path).stem
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"outputs/{input_stem}_image_to_3d_{timestamp}.glb"
    
    with open(output_filename, "wb") as f:
        f.write(output_bytes)
    print(f"Success! Generated GLB asset saved to {output_filename} ({len(output_bytes)} bytes)")

Sending request to Image-to-3D secure endpoint...
Success! Generated GLB asset saved to outputs/img5_image_to_3d_20260628_155535.glb (31889444 bytes)


### Step 4: Verify Security (Unauthorized Calls)

In [ ]:
print("--- Test: Querying WITHOUT credentials ---")
response = requests.post(ENDPOINT_URL, json={"image_base64": ""})
print(f"Status: {response.status_code} (Expected: 401)")
print(f"Message: {response.text}")